<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Solution: Deliver a reusable dataset

**[Deliver a reusable dataset](https://learning.nextia-ai.com/courses/sql/m06/deliver-a-reusable-dataset/)** · SQL and Data Preparation for AI · Module 6, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** put the whole preparation behind one command that starts from a clean snapshot. You prove with SHA-256 checksums that two runs give the same files, and you make the pipeline stop with a clear message when the export's schema changes.

| | |
|---|---|
| **Time** | About 55 minutes |
| **Needs** | An internet connection for the setup cells and for one small file. No account. The first cell installs pandas 3.0.6. |
| **You should know** | The data contract, from [Validation rules](https://learning.nextia-ai.com/courses/sql/m06/validation-rules/). The lineage records and the dataset card, from [Track lineage](https://learning.nextia-ai.com/courses/sql/m06/track-lineage/). |
| **You do not need** | Your local project. The setup below builds the database and the files of Modules 3 to 5 here, and downloads `validate.py` and `get_data.py`. |
| **Tested** | Python 3.14.6 with pandas 3.0.6, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every script, task and check of the lesson, and the module practice. The [lesson page](https://learning.nextia-ai.com/courses/sql/m06/deliver-a-reusable-dataset/) has the full explanations and the knowledge checks and module check that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M06-L03-deliver-a-reusable-dataset/deliver-a-reusable-dataset-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs `pandas` 3.0.6 in the versions that the course uses. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6
import pandas; print("pandas", pandas.__version__)

## Setup: get the practice data

Run the next cell. It downloads the three files of the Larkfield help-desk export (about 200 KB), checks each file's checksum, and builds the practice database, `larkfield.db`. It works in a folder `ticket-data`, laid out like your project from Module 1, so the code of the lesson runs here unchanged. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: customers 240, tickets 1417, outcomes 1874`. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-data` (in Colab: the **Files** panel on the left), then run the cell again.

This is the same database that `get_data.py` builds in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the Larkfield export, check it, and build larkfield.db in
# ticket-data/. Standard library only: the same steps as get_data.py.
import csv, hashlib, json, os, sqlite3, subprocess, sys, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/"
FILES = {
    "customers.csv": "7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993",
    "tickets.csv": "9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28",
    "outcomes.csv": "67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653",
}
# The notebook works in its own folder ticket-data/, marked with a file, so it
# never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-data").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, expected):
    path = PROJECT / "data" / "raw" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-data and run again.")
    return path

DB = PROJECT / "larkfield.db"
if not DB.exists():
    paths = {name: fetch(name, sha) for name, sha in FILES.items()}
    build = sqlite3.connect(DB)
    build.executescript("""
        CREATE TABLE customers (customer_id TEXT, segment TEXT, region TEXT, joined_on TEXT);
        CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, created_at TEXT, channel TEXT, team TEXT,
            priority INTEGER, order_value REAL, word_count INTEGER, first_reply_minutes INTEGER,
            priority_now INTEGER, closed_at TEXT);
        CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, recorded_at TEXT, status TEXT,
            resolution_code TEXT, csat INTEGER);
        CREATE TABLE snapshot (name TEXT, value TEXT);
        INSERT INTO snapshot VALUES ('source', 'Larkfield help-desk export (synthetic, Nextia Learning C04)'),
            ('exported_at', '2026-07-01 06:00:00'), ('data_version', '1.0');
    """)
    for name, path in paths.items():
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        marks = ", ".join("?" for _ in rows[0])
        build.executemany(f"INSERT INTO {name[:-4]} ({', '.join(rows[0])}) VALUES ({marks})",
                          [[None if v == "" else v for v in row] for row in rows[1:]])
    build.commit()
    build.close()

os.chdir(PROJECT)  # work in the project folder, as in the lesson
for folder in ("queries", "scripts", "reports"):
    Path(folder).mkdir(exist_ok=True)
# Read-only, as q.py: a query cannot change the data by mistake.
con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "tickets", "outcomes")]
print("Ready: customers {}, tickets {}, outcomes {}".format(*counts))

## Setup: rebuild the earlier lessons' files

This lesson starts from files that you made in earlier lessons: the extract `data/extract/` from [Reproducible extraction](https://learning.nextia-ai.com/courses/sql/m03/reproducible-extraction/), the clean tables `data/clean/` from [Handle messy values](https://learning.nextia-ai.com/courses/sql/m04/handle-messy-values/), the modelling table `data/model/tickets_model.csv` from [Features and target timing](https://learning.nextia-ai.com/courses/sql/m05/features-and-target-timing/), the train, validation and test sets `data/model/` from [Choose a split](https://learning.nextia-ai.com/courses/sql/m05/choose-a-split/). The next cell downloads the course's finished scripts ([C04/pipeline](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C04/pipeline)) into `ticket-data` and runs them in order, so you have the same files here. It takes about 20 seconds.

You should see one line per step, ending with `split.py: done`. On your computer, your own scripts from those lessons make the same files.

In [ ]:
# Setup: rebuild the files of the earlier lessons with the course's scripts.
STEPS = ["extract", "clean", "build_features", "split"]
for file in ["queries/extract_tickets.sql", "queries/extract_outcomes.sql"] + [f"scripts/{s}.py" for s in STEPS]:
    if not Path(file).exists():
        download(LABS + "pipeline/" + file, Path(file))
for step in STEPS:
    result = subprocess.run([sys.executable, f"scripts/{step}.py"], capture_output=True, text=True)
    if result.returncode:
        raise SystemExit(f"{step}.py failed:\n{result.stdout}{result.stderr}")
    print(f"{step}.py: done")

## Setup: the helpers

The next cell defines small functions that the lesson uses. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.
- `expect(what, yours, expected)` compares one of your numbers with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        # Show NULL as NULL, as q.py and the boxes on the lesson page do.
        table = pd.DataFrame(shown, columns=columns, dtype=object)
        display(table.where(table.notna(), "NULL"))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    try:
        got_columns, got = run(query, db)
    except sqlite3.Error as error:
        print(f"Not yet: the query has an error: {error}")
        return
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    same = abs(yours - expected) <= tolerance * max(1, abs(expected)) if isinstance(expected, float) else yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours}, which is not the expected value.")

## Setup: the earlier scripts, and a helper

This lesson also uses two files that the cell above did not download:

- `scripts/validate.py`, the contract from [Validation rules](https://learning.nextia-ai.com/courses/sql/m06/validation-rules/#the-contract-in-validatepy);
- `get_data.py`, which builds `larkfield.db` from the raw files, as in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database).

The next cell downloads both from the labs repository. It also defines `python()`. It runs a script as `python …` does in a terminal, prints the output, and then prints the **exit code**: 0 means success, any other number means failure.

You should see `All 13 rules passed.` and `[exit code 0]`.

In [ ]:
import json, shutil

def python(*args, cwd=None):
    """Run a Python script as `python <args>` in a terminal. Print its output and its exit code."""
    result = subprocess.run([sys.executable, *args], capture_output=True, text=True, cwd=cwd)
    print((result.stdout + result.stderr).rstrip())
    print(f"[exit code {result.returncode}]")
    return result

for file, url in [("scripts/validate.py", "pipeline/scripts/validate.py"), ("get_data.py", "get_data.py")]:
    if not Path(file).exists():
        download(LABS + url, Path(file))
contract = python("scripts/validate.py")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. What "reusable" means

Priya wants to make the files again: next month, on her computer, after a fix. Mei gives three promises:

1. **One command.** The whole preparation runs with one command, from a clean snapshot.
2. **Same input, same output.** Two runs on the same snapshot give the same files, byte for byte. A process with this property is **reproducible**.
3. **Changed input, clear failure.** When the export changes in a way the scripts do not expect, the run stops and says what changed.

You prove promise 2 with **checksums**, not by eye.

> **Predict.** Are the first two checksums equal? And the third? Then run the cell.

In [ ]:
import hashlib

print(hashlib.sha256(b"T-1,email,2\n").hexdigest())
print(hashlib.sha256(b"T-1,email,2\n").hexdigest())
print(hashlib.sha256(b"T-1,Email,2\n").hexdigest())

> **Check.** The first two lines are both `aba761b1…7ee2`. The third, with one capital letter, is `ebd66142…b550`: completely different. So if two runs give the same checksum for `train.csv`, the two files are identical.

## 2. The input check

`validate.py` checks the **output**. A new export can also break the **input**: a renamed column, or numbers written as text. The pipeline must find that before it loads the data. The check is `scripts/check_input.py`, with the Python standard library only. The [lesson page](https://learning.nextia-ai.com/courses/sql/m06/deliver-a-reusable-dataset/#the-input-check) explains each part.

Run the cell to write the script. You should see `Writing scripts/check_input.py`.

In [ ]:
%%writefile scripts/check_input.py
"""Check the raw export files before anything loads them: each file must have
the expected columns, and each value must be readable as its column's type.

    python scripts/check_input.py
    python scripts/check_input.py --tickets data/raw/tickets-2026-08-01.csv

This check looks at the shape of the data only. Messy values (spellings,
duplicates, sentinels) are the cleaning step's job. Exit code 1 on failure.
"""

import argparse
import csv
import sys
from datetime import datetime
from pathlib import Path

# The schema of the export of 2026-07-01: column name -> type.
SCHEMAS = {
    "customers": {"customer_id": "text", "segment": "text", "region": "text", "joined_on": "date"},
    "tickets": {
        "ticket_id": "text", "customer_id": "text", "created_at": "datetime", "channel": "text",
        "team": "text", "priority": "integer", "order_value": "number", "word_count": "integer",
        "first_reply_minutes": "integer", "priority_now": "integer", "closed_at": "datetime",
    },
    "outcomes": {
        "outcome_id": "text", "ticket_id": "text", "recorded_at": "datetime", "status": "text",
        "resolution_code": "text", "csat": "integer",
    },
}

# Each type has a function that raises ValueError for a value it cannot read.
READERS = {
    "text": str,
    "integer": int,
    "number": float,
    "date": lambda v: datetime.strptime(v, "%Y-%m-%d"),
    "datetime": lambda v: datetime.strptime(v, "%Y-%m-%d %H:%M:%S"),
}


def check_file(path, schema):
    """Return a list of problems. An empty list means the file is OK."""
    if not path.exists():
        return [f"file not found: {path}"]
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader, [])
        rows = list(reader)

    problems = []
    missing = [c for c in schema if c not in header]
    extra = [c for c in header if c not in schema]
    for column in missing:
        problems.append(f"missing column: {column}")
    for column in extra:
        hint = " (was a column renamed?)" if missing else ""
        problems.append(f"unexpected column: {column}{hint}")

    # An empty value is a missing value. It is allowed here; only values that
    # are present must have the right type.
    for column, kind in schema.items():
        if column not in header or kind == "text":
            continue
        i = header.index(column)
        bad = []
        for line, row in enumerate(rows, start=2):  # line 1 is the header
            value = row[i]
            if value == "":
                continue
            try:
                READERS[kind](value)
            except ValueError:
                bad.append((line, value))
        if bad:
            line, value = bad[0]
            present = sum(1 for row in rows if row[i] != "")
            problems.append(f"{column}: {len(bad)} of {present} values are not a valid {kind}"
                            f" (first: line {line}, {value!r})")
    return problems


def main():
    parser = argparse.ArgumentParser()
    for name in SCHEMAS:
        parser.add_argument(f"--{name}", default=f"data/raw/{name}.csv", type=Path)
    args = parser.parse_args()

    failed = False
    for name, schema in SCHEMAS.items():
        path = getattr(args, name)
        problems = check_file(path, schema)
        print(f"{'PASS' if not problems else 'FAIL'}  {path}")
        for problem in problems:
            print(f"      {problem}")
        failed = failed or bool(problems)

    if failed:
        print("The input does not match the expected schema. Do not load it.")
        sys.exit(1)
    print("The input matches the expected schema.")


if __name__ == "__main__":
    main()

Run it on the export of 2026-07-01. You should see 3 `PASS` lines, `The input matches the expected schema.` and `[exit code 0]`.

In [ ]:
check = python("scripts/check_input.py")

The check looks at the header and the types only. Spellings such as `Email` are not schema problems: `clean.py` handles them, and `validate.py` checks the result. Each check has one job.

## 3. The one command

`run_pipeline.py` runs the six steps in order and stops at the first step that fails. At the end, it writes `data/model/MANIFEST.json`, with a SHA-256 checksum and a row count for every output file. Four decisions make it safe to run again:

- **Delete old outputs first**, so an old file never looks like today's result.
- **Each step is a separate program.** `subprocess.run` reads its exit code.
- **Content apart from time.** The checksums are in `outputs`; the run times are in `run`.
- **One fingerprint.** `outputs_fingerprint` is a checksum of all the output checksums together.

Run the cell to write the script. It goes in the project folder, not in `scripts/`. You should see `Writing run_pipeline.py`.

In [ ]:
%%writefile run_pipeline.py
"""Run the whole preparation, in order, from larkfield.db to a validated
dataset. Stop at the first step that fails. At the end, write
data/model/MANIFEST.json with a SHA-256 checksum of every output file.

    python get_data.py --reset     start from a clean snapshot
    python run_pipeline.py

    python run_pipeline.py --tickets data/raw/tickets-2026-08-01.csv
                                   check a new export before you load it
"""

import argparse
import csv
import hashlib
import json
import shutil
import sqlite3
import subprocess
import sys
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

STEPS = [
    "scripts/check_input.py",
    "scripts/extract.py",
    "scripts/clean.py",
    "scripts/build_features.py",
    "scripts/split.py",
    "scripts/validate.py",
]
# Folders that only the pipeline writes. They are deleted before a run, so an
# old file can never look like the output of this run.
OUTPUT_FOLDERS = [Path("data/extract"), Path("data/clean"), Path("data/model")]
OUTPUTS = [
    "data/extract/tickets.csv", "data/extract/outcomes.csv",
    "data/clean/tickets.csv", "data/clean/quarantine.csv", "data/clean/outcomes.csv",
    "data/model/tickets_model.csv", "data/model/train.csv", "data/model/valid.csv", "data/model/test.csv",
    "data/model/build_summary.json", "data/model/split_manifest.json",
    "reports/cleaning.json", "reports/validation.json",
]
MANIFEST = Path("data/model/MANIFEST.json")


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def describe(path):
    """Checksum, and the number of data rows for a CSV file."""
    entry = {"sha256": sha256(path)}
    if path.endswith(".csv"):
        with open(path, newline="", encoding="utf-8") as f:
            entry["rows"] = sum(1 for _ in csv.reader(f)) - 1  # minus the header
    return entry


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--tickets", default="data/raw/tickets.csv",
                        help="the raw tickets file for the input check")
    args = parser.parse_args()

    if not Path("larkfield.db").exists():
        sys.exit("larkfield.db not found. Run: python get_data.py")
    for folder in OUTPUT_FOLDERS:
        shutil.rmtree(folder, ignore_errors=True)

    started = datetime.now(timezone.utc)
    for number, step in enumerate(STEPS, start=1):
        command = [sys.executable, step]  # the same Python that runs this script
        if step == "scripts/check_input.py":
            command += ["--tickets", args.tickets]
        name = Path(step).stem
        result = subprocess.run(command, capture_output=True, text=True)
        if result.returncode != 0:
            print(f"{number}/{len(STEPS)} {name}: FAILED\n")
            print((result.stdout + result.stderr).rstrip())
            print(f"\nStopped at step {number} ({name}). No MANIFEST.json was written: this run delivered no dataset.")
            sys.exit(1)
        print(f"{number}/{len(STEPS)} {name}: ok")

    # The extraction manifest says which snapshot and which selection the
    # dataset comes from. Its time stamp goes in "run", with this run's.
    extract = json.loads(Path("data/extract/manifest.json").read_text(encoding="utf-8"))
    outputs = {path: describe(path) for path in OUTPUTS}
    manifest = {
        "dataset": "Larkfield ticket escalation dataset",
        "source": extract["source"],
        "selection": extract["selection"],
        "queries": {f["query"]: f["query_sha256"] for f in extract["files"]},
        "steps": STEPS,
        "outputs": outputs,
        # One checksum for all outputs together: equal outputs give an
        # equal fingerprint, whatever the time of the run.
        "outputs_fingerprint": hashlib.sha256(json.dumps(outputs, sort_keys=True).encode()).hexdigest(),
        "run": {
            "started_at": started.isoformat(timespec="seconds"),
            "extracted_at": extract["extracted_at"],
            "python": sys.version.split()[0],
            "sqlite": sqlite3.sqlite_version,
            "pandas": version("pandas"),
        },
    }
    MANIFEST.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    model = outputs["data/model/tickets_model.csv"]["rows"]
    parts = ", ".join(f"{p} {outputs[f'data/model/{p}.csv']['rows']}" for p in ["train", "valid", "test"])
    print(f"Done: {model} rows ({parts}).")
    print(f"Wrote {MANIFEST}. Outputs fingerprint: {manifest['outputs_fingerprint']}")


if __name__ == "__main__":
    main()

## 4. Worked example: run it twice from a clean snapshot

Mei starts from a clean snapshot: `get_data.py --reset` deletes the database and builds it again from the checked CSV files. The first line of the cell closes the notebook's connection to the database, because the script deletes the file.

Run the cell. The last line is `Built larkfield.db. Check: customers 240, tickets 1417, outcomes 1874.`

In [ ]:
con.close()  # the notebook's read-only connection: get_data.py deletes the file
reset = python("get_data.py", "--reset")

Now the one command.

> **Predict.** How many steps does it run, and how many rows are in train, valid and test? Then run the cell.

In [ ]:
first = python("run_pipeline.py")

> **Check.** You should see six `ok` lines, `Done: 1369 rows (train 927, valid 227, test 215).` and the outputs fingerprint. On macOS with pandas 3.0.6, the fingerprint was `2b2460321543ffb73fe9d5b04fd8f5a9d77ee990fff37823b0855b911ba4b855`. Another system can give another fingerprint. The promise is that **your** two runs match.

Look at part of the manifest. Run the cell.

In [ ]:
manifest = json.loads(Path("data/model/MANIFEST.json").read_text(encoding="utf-8"))
print(json.dumps({"data/model/train.csv": manifest["outputs"]["data/model/train.csv"]}, indent=2))
print("outputs_fingerprint:", manifest["outputs_fingerprint"])
print(json.dumps(manifest["run"], indent=2))

> **Check.** `train.csv` has 927 rows and its checksum. `run` has the two times of this run and the versions of Python, SQLite and pandas.

Keep a copy of the manifest, as `first-run.json`. Then run the pipeline again. Run the cell.

In [ ]:
shutil.copy("data/model/MANIFEST.json", "first-run.json")
second = python("run_pipeline.py")

**The reproducibility check.** Compare the two manifests: the fingerprints, the `outputs` sections, and every line.

> **Predict.** Which lines of the two manifests are different? Then run the cell.

In [ ]:
import difflib

a = json.loads(Path("first-run.json").read_text(encoding="utf-8"))
b = json.loads(Path("data/model/MANIFEST.json").read_text(encoding="utf-8"))
print("first run: ", a["outputs_fingerprint"])
print("second run:", b["outputs_fingerprint"])
print("same outputs" if a["outputs"] == b["outputs"] else "different outputs")
print()
for line in difflib.unified_diff(Path("first-run.json").read_text(encoding="utf-8").splitlines(),
                                 Path("data/model/MANIFEST.json").read_text(encoding="utf-8").splitlines(),
                                 "first-run.json", "MANIFEST.json", n=0, lineterm=""):
    print(line)

> **Check.** The two fingerprints are equal, and you see `same outputs`. The line-by-line comparison shows only `started_at` and `extracted_at`: lines that start with `-` are in the first run, lines that start with `+` in the second. If both runs started in the same second, you see no lines at all. All 13 output files are identical. The pipeline is reproducible.

Run the check cell.

In [ ]:
if a["outputs_fingerprint"] == b["outputs_fingerprint"] and a["outputs"] == b["outputs"]:
    print("Check passed: two runs from the same snapshot gave the same 13 output files.")
else:
    different = [path for path in a["outputs"] if a["outputs"][path] != b["outputs"].get(path)]
    print("Not yet: the outputs are different:", ", ".join(different))

## 5. Your turn: the output checksums for the dataset card

Mei adds the checksums of the four files in `data/model/` to `DATASET.md`, so that Priya can check the files she receives.

> **Your turn.** Complete the loop: add one line per file to `lines`, in the form ``| `data/model/train.csv` | 927 | e83c…77b8 |``. Read the rows and the checksum from the manifest; do not type them. Write the number of rows with a thousands separator: `f"{rows:,}"` gives `1,369`. Run the cell, then the check cell.

In [ ]:
manifest = json.loads(Path("data/model/MANIFEST.json").read_text(encoding="utf-8"))
lines = ["| File | Rows | SHA-256 |", "|---|---|---|"]
for name in ["tickets_model", "train", "valid", "test"]:
    path = f"data/model/{name}.csv"
    entry = manifest["outputs"][path]
    lines.append(f"| `{path}` | {entry['rows']:,} | {entry['sha256']} |")
checksums = "\n".join(lines)
print(checksums)

In [ ]:
expected_lines = [f"| `data/model/{n}.csv` | {manifest['outputs'][f'data/model/{n}.csv']['rows']:,} | "
                  f"{manifest['outputs'][f'data/model/{n}.csv']['sha256']} |" for n in ["tickets_model", "train", "valid", "test"]]
mine = checksums.splitlines()[2:]
if mine == expected_lines:
    print("Check passed.")
else:
    print(f"Not yet: your table has {len(mine)} file line(s); it needs 4, one per file, in the order of the loop, "
          "with the path in backticks, the rows with a thousands separator, and the checksum.")

Run the cell to add the section to `DATASET.md`. If the file is not there (the card is from [Track lineage](https://learning.nextia-ai.com/courses/sql/m06/track-lineage/#worked-example-the-dataset-card)), the cell makes a new file with this section only.

In [ ]:
section = f"""

## Output checksums (SHA-256)

{checksums}

Outputs fingerprint (all 13 output files): `{manifest['outputs_fingerprint']}`.
"""
with open("DATASET.md", "a", encoding="utf-8") as card:
    card.write(section)
print(section)

## 6. Your turn: point the pipeline at the August export

The help desk was updated in July. The next export, of 2026-08-01, has 40 new tickets. Before Mei loads it, she checks it. Run the cell to download it into `data/raw/`. You should see `40 tickets`.

In [ ]:
download(LABS + "data/tickets-2026-08-01.csv", Path("data/raw/tickets-2026-08-01.csv"))
with open("data/raw/tickets-2026-08-01.csv", newline="", encoding="utf-8") as f:
    print(sum(1 for _ in csv.reader(f)) - 1, "tickets")

> **Predict.** Which step stops, and which problems does the message name?

> **Your turn.** In a terminal, the command is `python run_pipeline.py --tickets data/raw/tickets-2026-08-01.csv`. Change the cell so that `python()` gets the option and the file as two more arguments. Run the cell, then the check cell.

In [ ]:
# Run the pipeline with the input check pointed at the August file.
august = python("run_pipeline.py", "--tickets", "data/raw/tickets-2026-08-01.csv")

In [ ]:
stopped = august.returncode == 1 and "1/6 check_input: FAILED" in august.stdout
named = all(word in august.stdout for word in ["contact_channel", "order_value", "$10.85"])
if stopped and named and not Path("data/model/MANIFEST.json").exists():
    print("Check passed: the run stopped at step 1, named the changes, and delivered no dataset.")
elif august.returncode == 0:
    print("Not yet: the run ended with exit code 0. The input check did not look at the August file.")
else:
    print("Not yet: the run failed, but not at the input check with the August file.")

> **Check.** You should see `1/6 check_input: FAILED`, then `missing column: channel`, `unexpected column: contact_channel (was a column renamed?)` and `order_value: 34 of 34 values are not a valid number (first: line 2, '$10.85')`. The last lines are `Stopped at step 1 (check_input). No MANIFEST.json was written: this run delivered no dataset.` and `[exit code 1]`.

34 of 40 tickets have an order value; the other 6 are account tickets with no order, so their empty value is allowed. The run deleted the old outputs and delivered nothing: there is no `data/model/` folder. Run the cell to see.

In [ ]:
print("data/model exists:", Path("data/model").exists())

> **Check.** You should see `data/model exists: False`.

The fix is not in the check. Mei asks Grace about the update, then changes `clean.py` and the schema in `check_input.py` on purpose. The result is a new dataset version, recorded in the **Changes** table of `DATASET.md`.

## 7. Failure case: a changed export with no input check

**Symptom:** a script that worked last month gives strange results, or an error far from the cause.

**Cause:** the schema changed, and nothing checked it before the data was used. pandas reads the August file with no warning.

> **Predict.** What type does pandas give `order_value`? How many rows equal −1? Then run the cell.

In [ ]:
august_df = pd.read_csv("data/raw/tickets-2026-08-01.csv")
print(august_df["order_value"].head(3).tolist())
print(august_df["order_value"].dtype)
print((august_df["order_value"] == -1).sum())
try:
    august_df["channel"]
except KeyError as error:
    print("KeyError:", error)
try:
    pd.read_csv("data/raw/tickets-2026-08-01.csv", dtype={"order_value": "Float64"})
except ValueError as error:
    print("ValueError:", error)

> **Check.** You should see `['$10.85', '$25.58', '$91.70']`, a text type (`str`), `0`, `KeyError: 'channel'`, and a `ValueError` that names the value `$10.85`.

`order_value` is now text. The comparison with −1 does not fail; it is false for every row, so the "unknown" rule finds nothing. The `KeyError` names a column, but not the reason. The `ValueError` names a value, but not the column or the file.

**Fix:** run `scripts/check_input.py` on every new export before you load it, and keep it as step 1 of `run_pipeline.py`. When it fails, change the schema and the cleaning on purpose, and make a new dataset version.

**In AI work:** a renamed column can turn a feature into "all missing", and a model in production keeps making predictions, only worse. An input check turns that into a clear stop.

## 8. Module practice: the steps by hand, with their counts

Before she trusts the one command, Mei runs each step by hand once, from a clean snapshot. Each count must match the lineage records.

> **Predict.** Which step changes the number of ticket rows the most? Then run the cell. It runs the seven commands of the lesson's table and prints each exit code.

In [ ]:
steps = [["get_data.py", "--reset"], ["scripts/check_input.py"], ["scripts/extract.py"], ["scripts/clean.py"],
         ["scripts/build_features.py"], ["scripts/split.py"], ["scripts/validate.py"]]
for step in steps:
    result = subprocess.run([sys.executable, *step], capture_output=True, text=True)
    last = result.stdout.strip().splitlines()[-1]
    shown = "" if last == "}" else f"   {last}"  # clean.py and split.py end with their JSON summary
    print(f"python {' '.join(step):<26} exit code {result.returncode}{shown}")

> **Check.** You should see exit code 0 seven times. The last line of `validate.py` is `All 13 rules passed.`

Now read the counts from the records, and check that each follows from the one before it. Run the cell.

In [ ]:
extract = json.loads(Path("data/extract/manifest.json").read_text(encoding="utf-8"))
cleaning = json.loads(Path("reports/cleaning.json").read_text(encoding="utf-8"))["tickets"]
build = json.loads(Path("data/model/build_summary.json").read_text(encoding="utf-8"))
parts = json.loads(Path("data/model/split_manifest.json").read_text(encoding="utf-8"))["parts"]
print("database:", extract["ticket_rows_in_source"], "- not selected", extract["ticket_rows_not_selected"],
      "=", extract["files"][0]["rows"])
print("extract: ", cleaning["input_rows"], "- duplicates", cleaning["exact_duplicates_removed"],
      "- quarantined", cleaning["quarantined"], "=", cleaning["clean_rows"])
print("clean:   ", build["clean_tickets"], "- too recent", build["too_recent_for_label"], "=", build["rows"])
print("split:   ", " + ".join(str(parts[p]["rows"]) for p in parts), "=", sum(parts[p]["rows"] for p in parts))

> **Check.** You should see 1417 − 1 = 1416, 1416 − 14 − 4 = 1398, 1398 − 29 = 1369, and 927 + 227 + 215 = 1369. `run_pipeline.py` runs the same steps, so it must give the same counts.

## 9. Your task: a fresh copy

Prove that the project runs anywhere, not only in this folder with its old files. You make a new folder, `ticket-data-fresh`, next to `ticket-data`, with **only the code and the raw data**. Then you run the two commands there.

On your computer, you also make a new virtual environment in the new folder and install `requirements.txt`. A notebook has one environment, so here you use the same one.

> **Your turn.** Complete the list `COPY`. It must have the code and the raw data: the two scripts in the project folder, the folders `queries` and `scripts`, and `data/raw`. Do not copy `larkfield.db`, `data/extract`, `data/clean`, `data/model` or `reports`: the pipeline makes them. Run the cell, then the next one.

In [ ]:
# Copy only the code and the raw data. Add the folders and files that are missing.
COPY = ["get_data.py", "run_pipeline.py", "queries", "scripts", "data/raw"]

In [ ]:
FRESH = Path("../ticket-data-fresh").resolve()
shutil.rmtree(FRESH, ignore_errors=True)
for item in COPY:
    target = FRESH / item
    target.parent.mkdir(parents=True, exist_ok=True)
    if Path(item).is_dir():
        shutil.copytree(item, target, ignore=shutil.ignore_patterns("__pycache__"))
    else:
        shutil.copy2(item, target)
print("copied:", sorted(str(p.relative_to(FRESH)) for p in FRESH.iterdir()))
built = python("get_data.py", cwd=FRESH)
fresh = python("run_pipeline.py", cwd=FRESH)

> **Check.** In the solution, you see `Built larkfield.db. …`, six `ok` lines and the same fingerprint as in section 4. If a step fails with `can't open file` or `No such file`, a file or folder is missing from `COPY`.

Run the check cell. It compares the fingerprint of the fresh copy with your first run.

In [ ]:
fresh_manifest = FRESH / "data/model/MANIFEST.json"
if not fresh_manifest.exists():
    print("Not yet: the fresh copy has no MANIFEST.json. Read the output above to find the step that failed.")
elif json.loads(fresh_manifest.read_text(encoding="utf-8"))["outputs_fingerprint"] == a["outputs_fingerprint"]:
    print("Check passed: the fresh copy gave the same fingerprint as the project folder.")
else:
    print("Not yet: the fresh copy gave a different fingerprint. Compare the outputs sections to find the first file that differs.")

Last, run the fresh copy with the August export. It must stop at step 1, as in section 6. Run the cell.

In [ ]:
fresh_august = python("run_pipeline.py", "--tickets", "data/raw/tickets-2026-08-01.csv", cwd=FRESH)

> **Check.** In the solution, you see `Stopped at step 1 (check_input).` and `[exit code 1]`. If `COPY` does not have `data/raw`, the August file is not in the fresh copy, and the message says `file not found`.

On your computer, write two sentences in `DATASET.md`, under **Changes** or in a new section **Reproducibility**: the date, the fingerprint, and "two environments gave the same fingerprint".

## Module check and recap

Answer the **knowledge checks** of the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/sql/m06/deliver-a-reusable-dataset/#module-check). They count toward your certificate when you are signed in and enrolled. The completion check asks for the evidence that you made here: two runs with the same fingerprint, and a run with the August export that stops with a clear message.

In this module you turned a set of scripts into a dataset that another person can trust and run again. `validate.py` checks the output. `DATASET.md` records the lineage. `check_input.py` stops a changed export before it is loaded. `run_pipeline.py` builds everything with one command, and `MANIFEST.json` proves with checksums that two runs give the same files.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Final assignment](https://learning.nextia-ai.com/courses/sql/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m06/deliver-a-reusable-dataset/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.